# Convert an inventory policy to mathematical notation

This notebook uses `policy_source_to_compact_math` for recognized policy families and `policy_source_to_math` for the general declarative renderer. The goal is mathematical notation rather than line-by-line algorithmic pseudocode: reductions become summations, branches become piecewise equations, and displayed constants are rounded to one decimal place.

In [1]:
from pathlib import Path
import sys

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "python_to_math" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not find the repository root containing the python_to_math package.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from python_to_math import policy_source_to_compact_math, policy_source_to_math

## Sample policy

The policy is stored as source text so it can be parsed directly. This compact renderer supports parameter variants of the recognized policy structure below. A structurally different policy needs its own compact renderer because producing an understandable summary requires policy-specific reasoning.

In [2]:
policy_source = r'''# sample_id: 004
# folder: deepseek-chat_poisson_L6_c1_5_50_plain_processed_scipy_15_default_e1-e2-m2_4_r1
# distribution: poisson_L6_c1_5
# generation: 8
# rank_in_population_file: 2
# objective: 1238.22
# test_objective: 1272.82
# is_top10_by_distribution: False
# is_final_generation: False
# table_motifs: inventory_position;pipeline_weighting;nonlinear_pipeline_composition;state_dependent_target;partial_adjustment;order_clipping
# extra_motifs: safety_stock_buffer;pipeline_demand_proxy;near_term_pipeline_focus;threshold_order_activation;integer_rounding
def compute_order_amount(on_hand_inventory, pipeline_orders):
    # Base stock level
    base_stock = 583.5806013968356  # OPT_PARAM: {"initial": 583.5806013968356, "min": 400, "max": 800, "type": "float"}
    
    # Safety stock adjustment
    safety_stock = 122.3200350131986  # OPT_PARAM: {"initial": 122.3200350131986, "min": 20, "max": 150, "type": "float"}
    
    # Demand anticipation factor
    anticipation_factor = 1.5  # OPT_PARAM: {"initial": 1.5, "min": 0.5, "max": 1.5, "type": "float"}
    
    # Order smoothing
    smoothing_factor = 0.1  # OPT_PARAM: {"initial": 0.1, "min": 0.1, "max": 0.8, "type": "float"}
    
    # Order bounds
    min_order = 96.64831031968099  # OPT_PARAM: {"initial": 96.64831031968099, "min": 20, "max": 100, "type": "float"}
    max_order = 200.0  # OPT_PARAM: {"initial": 200.0, "min": 100, "max": 300, "type": "float"}
    
    # Pipeline urgency adjustment
    urgency_factor = 0.5  # OPT_PARAM: {"initial": 0.5, "min": 0.5, "max": 1.5, "type": "float"}
    urgency_threshold = 50.0  # OPT_PARAM: {"initial": 50.0, "min": 10, "max": 150, "type": "float"}
    
    # Lead time demand coverage
    coverage_periods = 0.5  # OPT_PARAM: {"initial": 0.5, "min": 0.5, "max": 2.0, "type": "float"}
    
    # NEW: Cost-ratio adjustment factor
    cost_ratio_factor = 0.5  # OPT_PARAM: {"initial": 0.5, "min": 0.5, "max": 1.5, "type": "float"}
    
    # NEW: Predictive ordering based on pipeline pattern recognition
    pattern_weight = 0.3  # OPT_PARAM: {"initial": 0.3, "min": 0.1, "max": 0.8, "type": "float"}
    
    # NEW: Dynamic risk adjustment based on inventory position
    risk_sensitivity = 0.1  # OPT_PARAM: {"initial": 0.1, "min": 0.1, "max": 1.5, "type": "float"}
    
    # Calculate inventory position
    inventory_position = on_hand_inventory + sum(pipeline_orders)
    
    # Estimate near-term demand using weighted average of recent pipeline arrivals
    weighted_demand_estimate = 0
    total_weight = 0
    L = len(pipeline_orders)
    
    for i, q in enumerate(pipeline_orders):
        # Weight decreases for older pipeline orders
        weight = (L - i) / L
        weighted_demand_estimate += q * weight
        total_weight += weight
    
    if total_weight > 0:
        avg_weighted_demand = weighted_demand_estimate / total_weight
    else:
        avg_weighted_demand = 100.0  # Default estimate
    
    # NEW: Pattern recognition - detect if pipeline shows increasing/decreasing trend
    if L >= 3:
        recent_trend = 0
        for i in range(L-1):
            if pipeline_orders[i+1] > pipeline_orders[i]:
                recent_trend += 1
            elif pipeline_orders[i+1] < pipeline_orders[i]:
                recent_trend -= 1
        
        # Adjust demand estimate based on trend
        trend_factor = 1.0 + (recent_trend / L) * pattern_weight
        pattern_adjusted_demand = avg_weighted_demand * trend_factor
    else:
        pattern_adjusted_demand = avg_weighted_demand
    
    # Adjust base stock based on demand anticipation
    adjusted_base_stock = base_stock + anticipation_factor * (pattern_adjusted_demand - 100.0)
    
    # Apply pipeline urgency adjustment
    near_pipeline = sum(pipeline_orders[:min(2, L)])
    if near_pipeline < urgency_threshold:
        urgency_adjustment = 1.0 + (urgency_threshold - near_pipeline) / urgency_threshold * (urgency_factor - 1.0)
        adjusted_base_stock *= urgency_adjustment
    
    # Ensure lead time demand coverage
    lead_time_demand = pattern_adjusted_demand * L
    coverage_adjustment = lead_time_demand * coverage_periods
    final_base_stock = max(adjusted_base_stock, coverage_adjustment)
    
    # NEW: Dynamic risk adjustment based on current inventory position
    # Higher inventory reduces risk sensitivity, lower inventory increases it
    risk_adjustment = 1.0 + (risk_sensitivity * (1.0 - min(1.0, inventory_position / final_base_stock)))
    final_base_stock *= risk_adjustment
    
    # Adjust for cost ratio (p/h = 5/1 = 5)
    # Higher p/h ratio favors higher inventory to avoid stockouts
    cost_adjusted_base_stock = final_base_stock * cost_ratio_factor
    
    # Add safety stock
    target_inventory_position = cost_adjusted_base_stock + safety_stock
    
    # Calculate raw order
    raw_order = max(0, target_inventory_position - inventory_position)
    
    # Apply smoothing
    smoothed_order = smoothing_factor * raw_order + (1 - smoothing_factor) * min_order
    
    # Apply bounds
    bounded_order = max(min_order, min(max_order, smoothed_order))
    
    # Round to nearest integer
    order_amount = int(round(bounded_order))
    
    return order_amount
'''

## Generate a compact mathematical representation

The resulting LaTeX introduces a few readable intermediate quantities and ends with one boxed order function. It is deliberately not a line-by-line translation.

In [3]:
math_policy = policy_source_to_compact_math(policy_source)
print(math_policy)

% Compact mathematical representation of the recognized inventory policy.
% Policy constants are rounded to one decimal place for readability.
% round_even means Python's nearest-integer rounding, with ties rounded to even.
\[
\begin{aligned}
&\text{Inputs: } h=\text{on-hand inventory},\quad \mathbf{q}=(q_1,\ldots,q_L)=\text{pipeline orders}. \\
&I = h+\sum_{i=1}^{L}q_i \qquad \text{(inventory position)}. \\[3pt]
&\bar d =\begin{cases}\displaystyle \frac{2}{L(L+1)}\sum_{i=1}^{L}(L-i+1)q_i, & L>0, \\100, & L=0\end{cases}\qquad \text{(weighted pipeline-demand estimate)}. \\[6pt]
&s=\sum_{i=1}^{L-1}\operatorname{sgn}(q_{i+1}-q_i)\qquad \text{(pipeline trend score)}. \\[3pt]
&d=\begin{cases}\bar d\left(1+\dfrac{0.3}{L}s\right), & L\ge 3, \\\bar d, & L<3\end{cases}\qquad \text{(trend-adjusted demand estimate)}. \\[6pt]
&n=\sum_{i=1}^{\min(2,L)}q_i\qquad \text{(near-term pipeline)}. \\[3pt]
&u(n)=\begin{cases}1+\dfrac{50.0-n}{50.0}\left(0.5-1\right), & n<50.0, \\1, & n\ge 50.0\end{cases}\qqu

## Render the LaTeX

In a Jupyter environment, this cell displays the generated policy as typeset mathematics.

The main symbols are:

- $I$: inventory position after accounting for pipeline orders.
- $\bar d$ and $d$: weighted and trend-adjusted demand estimates.
- $n$ and $u(n)$: near-term pipeline quantity and urgency multiplier.
- $B$, $G$, and $R$: successive target-stock adjustments.
- $T$: target inventory position.
- $Q(h,\mathbf q)$: final rounded and clipped order quantity.

In [4]:
try:
    from IPython.display import Math, display
except ImportError:
    class Math(str):
        pass

    def display(obj):
        print(obj)

latex_body = math_policy[math_policy.index(r"\[") + 2 : math_policy.rindex(r"\]")]
display(Math(latex_body))


\begin{aligned}
&\text{Inputs: } h=\text{on-hand inventory},\quad \mathbf{q}=(q_1,\ldots,q_L)=\text{pipeline orders}. \\
&I = h+\sum_{i=1}^{L}q_i \qquad \text{(inventory position)}. \\[3pt]
&\bar d =\begin{cases}\displaystyle \frac{2}{L(L+1)}\sum_{i=1}^{L}(L-i+1)q_i, & L>0, \\100, & L=0\end{cases}\qquad \text{(weighted pipeline-demand estimate)}. \\[6pt]
&s=\sum_{i=1}^{L-1}\operatorname{sgn}(q_{i+1}-q_i)\qquad \text{(pipeline trend score)}. \\[3pt]
&d=\begin{cases}\bar d\left(1+\dfrac{0.3}{L}s\right), & L\ge 3, \\\bar d, & L<3\end{cases}\qquad \text{(trend-adjusted demand estimate)}. \\[6pt]
&n=\sum_{i=1}^{\min(2,L)}q_i\qquad \text{(near-term pipeline)}. \\[3pt]
&u(n)=\begin{cases}1+\dfrac{50.0-n}{50.0}\left(0.5-1\right), & n<50.0, \\1, & n\ge 50.0\end{cases}\qquad \text{(urgency multiplier)}. \\[6pt]
&B=u(n)\left[583.6+1.5(d-100)\right]. \\[3pt]
&G=\max\left(B,\;0.5Ld\right). \\[3pt]
&R=G\left[1+0.1\left(1-\min\left(1,\frac{I}{G}\right)\right)\right]. \\[3pt]
&T=0.5R+122.3\qquad \tex

## Test the declarative renderer

Sample `005` does not match one of the hand-simplified policy families. The general renderer still expresses it mathematically: accumulator loops become summations and branches become piecewise equations.

In [5]:
policy_source_005 = (repo_root / "python_to_math" / "sampled_policies" / "sample_005__normal_std30_L6_c1_2__gen02__rank02.py").read_text()

math_policy_005 = policy_source_to_math(policy_source_005)
assert r"\text{for }" not in math_policy_005
assert r"\operatorname{enumerate}" not in math_policy_005
assert r"\mathrm{weighted\_pipeline\_sum} = \sum_{i=0}^{L-1}" in math_policy_005
print(math_policy_005)

latex_body_005 = math_policy_005[math_policy_005.index(r"\[") + 2 : math_policy_005.rindex(r"\]")]
display(Math(latex_body_005))

% Mathematical policy generated from Python source.
% Notes:
% - Indices and slices retain Python's zero-based and negative-index semantics.
% - This is a structured mathematical transcription, not necessarily a single closed-form expression.
\[
\begin{aligned}
&\text{Inputs: } h=\text{on-hand inventory},\quad \mathbf{q}=(q_0,\ldots,q_{L-1}) \\
&\mathrm{weighted\_pipeline\_sum} = \sum_{i=0}^{L-1}\left(\left(q_{i} \cdot \left(1.7\right)^{\left(\left(L - i\right) - 1\right)}\right)\right) \\
&\mathrm{total\_weight} = \sum_{i=0}^{L-1}\left(\left(1.7\right)^{\left(\left(L - i\right) - 1\right)}\right) \\
&\mathrm{effective\_pipeline} = \begin{cases}\frac{\mathrm{weighted\_pipeline\_sum}}{\mathrm{total\_weight}}, & \mathrm{total\_weight} > 0 \\ 0, & \text{otherwise}\end{cases} \\
&\left.\begin{gathered}\mathrm{pipeline\_mean} = \frac{\sum_{i=0}^{L-1} q_i}{L} \\ \mathrm{variance} = \frac{\sum_{q \in \mathbf{q}} \left(\left(q - \mathrm{pipeline\_mean}\right)\right)^{2}}{L} \\ \mathrm{pipeline

## Test the policy family behind sample `001`

This is the case that previously fell back to algorithm-like LaTeX. The compact renderer now summarizes it as an actual policy formula with one-decimal constants.

In [6]:
policy_source_001 = (repo_root / "python_to_math" / "sampled_policies" / "sample_001__poisson_L6_c1_2__gen07__rank01.py").read_text()

math_policy_001 = policy_source_to_compact_math(policy_source_001)
assert r"\text{for }" not in math_policy_001
assert "852.0694799766587" not in math_policy_001
assert r"D_0(\mathbf{q})=852.1" in math_policy_001
print(math_policy_001)

latex_body_001 = math_policy_001[math_policy_001.index(r"\[") + 2 : math_policy_001.rindex(r"\]")]
display(Math(latex_body_001))

% Compact mathematical representation of the recognized inventory policy.
% Policy constants are rounded to one decimal place for readability.
% round_even means Python's nearest-integer rounding, with ties rounded to even.
% recent_demand_weight is omitted because it is assigned in the Python code but never used.
\[
\begin{aligned}
&\text{Inputs: } h=\text{on-hand inventory},\quad \mathbf{q}=(q_1,\ldots,q_L)=\text{pipeline orders}. \\
&[x]^+=\max(0,x). \\[3pt]
&H_L=\sum_{i=1}^{L}\frac{1}{i}. \\[3pt]
&e(\mathbf{q})=\begin{cases}\displaystyle \frac{\sum_{i=1}^{L}q_i/i}{H_L}, & L>0, \\0, & L=0\end{cases}\qquad \text{(reciprocal-weighted pipeline)}. \\[6pt]
&u(\mathbf{q})=\begin{cases}\displaystyle \frac{\sum_{i=\lfloor L/2\rfloor+1}^{L}q_i}{\sum_{i=1}^{L}q_i}, & L\ge 2 \text{ and } \sum_{i=1}^{L}q_i>0, \\0, & \text{otherwise}\end{cases}\qquad \text{(share of pipeline arriving later)}. \\[6pt]
&I(h,\mathbf{q})=h+e(\mathbf{q}). \\[3pt]
&D_0(\mathbf{q})=852.1\left(1+0.6u(\mathbf{q})\right).

## Test a bounded base-stock policy

Sample `003` has a simpler structure. The compact renderer combines the lead-time target and the base-stock cap into one understandable order function.

In [7]:
policy_source_003 = r'''# sample_id: 003
# folder: deepseek-chat_poisson_L6_c1_5_50_plain_processed_scipy_15_default_m2_10_r3
# distribution: poisson_L6_c1_5
# generation: 2
# rank_in_population_file: 6
# objective: 3336.88008
# test_objective: 3369.77227
# is_top10_by_distribution: False
# is_final_generation: False
# table_motifs: inventory_position;pipeline_weighting;state_dependent_target
# extra_motifs: safety_stock_buffer;pipeline_demand_proxy
def compute_order_amount(on_hand_inventory, pipeline_orders):
    base_stock = 697.9997255235969  # OPT_PARAM: {"initial": 697.9997255235969, "min": 10, "max": 1000, "type": "float"}
    safety_stock = 79.3000000000116  # OPT_PARAM: {"initial": 79.3000000000116, "min": 0, "max": 200, "type": "float"}
    demand_forecast = 149.9  # OPT_PARAM: {"initial": 149.9, "min": 50, "max": 150, "type": "float"}
    lead_time = len(pipeline_orders)
    
    # Calculate inventory position
    inventory_position = on_hand_inventory + sum(pipeline_orders)
    
    # Calculate expected demand during lead time
    expected_lead_time_demand = demand_forecast * lead_time
    
    # Calculate target inventory position
    target_inventory = expected_lead_time_demand + safety_stock
    
    # Calculate order amount
    order_amount = max(0, target_inventory - inventory_position)
    
    # Apply base stock as upper bound
    order_amount = min(order_amount, max(0, base_stock - inventory_position))
    
    return order_amount
'''

math_policy_003 = policy_source_to_compact_math(policy_source_003)
assert r"\boxed{Q(h,\mathbf{q})=" in math_policy_003
print(math_policy_003)

latex_body_003 = math_policy_003[math_policy_003.index(r"\[") + 2 : math_policy_003.rindex(r"\]")]
display(Math(latex_body_003))

% Compact mathematical representation of the recognized inventory policy.
% Policy constants are rounded to one decimal place for readability.
% [x]^+ means max(0, x).
\[
\begin{aligned}
&\text{Inputs: } h=\text{on-hand inventory},\quad \mathbf{q}=(q_1,\ldots,q_L)=\text{pipeline orders}. \\
&I(h,\mathbf{q})=h+\sum_{i=1}^{L}q_i\qquad \text{(inventory position)}. \\[3pt]
&T(L)=149.9L+79.3\qquad \text{(lead-time demand plus safety stock)}. \\[3pt]
&[x]^+=\max(0,x). \\[3pt]
&\boxed{Q(h,\mathbf{q})=\left[\min\left(698.0,T(L)\right)-I(h,\mathbf{q})\right]^+}.
\end{aligned}
\]

\begin{aligned}
&\text{Inputs: } h=\text{on-hand inventory},\quad \mathbf{q}=(q_1,\ldots,q_L)=\text{pipeline orders}. \\
&I(h,\mathbf{q})=h+\sum_{i=1}^{L}q_i\qquad \text{(inventory position)}. \\[3pt]
&T(L)=149.9L+79.3\qquad \text{(lead-time demand plus safety stock)}. \\[3pt]
&[x]^+=\max(0,x). \\[3pt]
&\boxed{Q(h,\mathbf{q})=\left[\min\left(698.0,T(L)\right)-I(h,\mathbf{q})\right]^+}.
\end{aligned}

